# 改进版每日交易系统 Runner

支持单线程/多线程/多进程模式，避免 baostock 访问限制问题。

## 并发模式说明
- **单线程模式 (single)**: 最稳定，避免所有并发问题
- **多线程模式 (thread)**: 在避免baostock限制的同时提供一定并发性
- **多进程模式 (process)**: 速度最快，但可能触发baostock限制

## 使用建议
1. 如果遇到 baostock 访问限制，请使用单线程模式
2. 需要一定并发性但避免限制时，使用多线程模式
3. 网络环境良好时，使用多进程模式获得最快速度

In [ ]:
# 导入必要的库
import sys
import os
from datetime import datetime

# 添加当前目录到路径
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

## 配置参数

请根据需要修改以下参数：

In [ ]:
# ========== 并发模式配置 ==========
CONCURRENCY_MODE = 'single'  # 可选: 'single', 'thread', 'process'
MAX_WORKERS = 1  # 工作进程/线程数

# ========== 数据更新配置 ==========
SKIP_UPDATE = False  # 是否跳过数据更新
UPDATE_ONLY = False  # 是否只更新数据
RECOMMEND_ONLY = False  # 是否只生成推荐
FORCE_WEEKLY = False  # 是否强制更新周线数据

# ========== 策略配置 ==========
STRATEGIES = ['weekly_mean_down', 'cross_ma50']  # 使用的策略
PRICE_CAP = 150.0  # 价格上限
MAX_HOLDINGS = 3  # 最大持仓数
TOP_N = 20  # 返回前N只推荐股票

# ========== 其他配置 ==========
INCLUDE_GEM = False  # 是否包含创业板股票
IGNORE_HOLDINGS = False  # 是否忽略当前持仓数量限制
CLEAR_POSITIONS = False  # 是否清空持仓记录
ENABLE_SLOW_FACTORS = False  # 是否启用耗时较长的复杂因子
FACTOR_PROGRESS = False  # 是否打印因子计算进度
SIMPLE_FILTER = False  # 是否使用简单过滤

# ========== 显示配置 ==========
SHOW_PROGRESS = True  # 是否显示进度条
DEBUG_MODE = False  # 是否启用调试模式

## 1. 数据更新

更新股票数据，支持不同的并发模式。

In [ ]:
if not RECOMMEND_ONLY and not SKIP_UPDATE:
    print(f"📊 开始更新每日数据 (模式: {CONCURRENCY_MODE}, 工作数: {MAX_WORKERS})...")
    print("-" * 60)
    
    # 导入更新函数
    from daily_update import update_daily_data
    
    # 更新数据
    update_daily_data(
        force_weekly=FORCE_WEEKLY,
        concurrency_mode=CONCURRENCY_MODE,
        max_workers=MAX_WORKERS
    )
    
    print("✅ 数据更新完成")
elif SKIP_UPDATE and not RECOMMEND_ONLY:
    print("⏭️ 已跳过数据更新，使用现有数据库数据")

# 如果只更新数据，则退出
if UPDATE_ONLY:
    print("✅ 数据更新任务完成")
    sys.exit(0)

## 2. 交易推荐

生成交易推荐，包括卖出建议和买入建议。

In [ ]:
if not UPDATE_ONLY:
    print("\n🎯 生成交易推荐...")
    print("=" * 80)
    
    # 导入必要的模块
    from daily_trading_system import (
        load_positions,
        save_positions,
        clear_positions,
        generate_sell_recommendations,
        generate_trading_recommendations,
        POSITIONS_FILE
    )
    
    # 清空持仓记录（如果配置）
    if CLEAR_POSITIONS:
        clear_positions()
        print(f"🧹 已清空持仓记录: {POSITIONS_FILE}")
    
    # 加载持仓
    positions = load_positions()
    
    # 生成卖出建议
    sell_df, status_df, positions = generate_sell_recommendations(positions)
    
    # 保存更新后的持仓
    save_positions(positions)
    
    # 显示当前持仓监控
    if status_df is not None and not status_df.empty:
        print("\n📦 当前持仓监控:")
        print("=" * 80)
        for idx, row in status_df.iterrows():
            print(f"{idx+1:2d}. {row['stock_code']:10s} | "
                  f"持仓: {row['shares']:6d}股 | "
                  f"成本: {row['entry_price']:6.2f} | "
                  f"现价: {row['last_price']:6.2f} | "
                  f"盈亏: {row['profit_rate']:6.1%} | "
                  f"持有: {row['holding_days']:3d}天 | "
                  f"状态: {row['status']}")
    
    # 显示卖出建议
    if sell_df is not None and not sell_df.empty:
        print("\n💸 今日卖出建议:")
        print("=" * 80)
        for idx, row in sell_df.iterrows():
            print(f"{idx+1:2d}. {row['stock_code']:10s} | "
                  f"持仓: {row['shares']:6d}股 | "
                  f"成本: {row['entry_price']:6.2f} | "
                  f"现价: {row['last_price']:6.2f} | "
                  f"盈亏: {row['profit_rate']:6.1%} | "
                  f"持有: {row['holding_days']:3d}天 | "
                  f"理由: {row['reason']}")
    else:
        print("\n✅ 今日无卖出建议")
    
    # 生成买入建议
    print("\n📈 今日买入建议:")
    print("=" * 80)
    
    # 根据策略生成推荐
    if 'cross_ma50' in STRATEGIES:
        print(f"🔍 使用 cross_ma50 策略 (价格上限: {PRICE_CAP}元)")
        # 这里可以调用具体的策略函数
        
    if 'weekly_mean_down' in STRATEGIES:
        print(f"🔍 使用 weekly_mean_down 策略")
        # 这里可以调用具体的策略函数
    
    # 调用交易推荐函数
    generate_trading_recommendations(
        exclude_gem=not INCLUDE_GEM,
        exclude_star=True,
        top_n=TOP_N,
        enable_slow_factors=ENABLE_SLOW_FACTORS,
        factor_progress=FACTOR_PROGRESS,
        simple_filter=SIMPLE_FILTER
    )
    
    print("\n✅ 交易推荐生成完成")

## 3. 快速命令

以下是一些常用的快速命令，可以直接运行：

In [ ]:
# 单线程模式（最稳定）
print("🚀 单线程模式运行:")
!python daily_trading_system.py --mode single --workers 1 --strategy weekly_mean_down cross_ma50 --price-cap {PRICE_CAP} --max-holdings {MAX_HOLDINGS}

In [ ]:
# 多线程模式（平衡）
print("🚀 多线程模式运行:")
!python daily_trading_system.py --mode thread --workers 3 --strategy weekly_mean_down cross_ma50 --price-cap {PRICE_CAP} --max-holdings {MAX_HOLDINGS}

In [ ]:
# 多进程模式（最快）
print("🚀 多进程模式运行:")
!python daily_trading_system.py --mode process --workers 6 --strategy weekly_mean_down cross_ma50 --price-cap {PRICE_CAP} --max-holdings {MAX_HOLDINGS}

## 4. 测试命令

测试不同并发模式的功能：

In [ ]:
# 测试并发模式
print("🧪 测试并发模式功能:")
!python test_concurrency_modes.py --mode single --skip-process

In [ ]:
# 简单连接测试
print("🔗 简单连接测试:")
!python test_simple.py

## 5. 使用指南

### 常见问题解决

1. **遇到 baostock 访问限制**：
   - 使用单线程模式：`--mode single --workers 1`
   - 降低并发数：`--workers 2`

2. **下载速度过慢**：
   - 尝试多线程模式：`--mode thread --workers 4`
   - 如果网络环境好，尝试多进程模式：`--mode process --workers 6`

3. **测试模式**：
   - 只处理少量股票：`--test`
   - 启用调试信息：`--debug`

### 推荐配置

| 场景 | 模式 | 工作数 | 说明 |
|------|------|--------|------|
| 稳定优先 | `single` | 1 | 避免所有并发问题 |
| 平衡配置 | `thread` | 3-4 | 在稳定性和速度之间平衡 |
| 性能优先 | `process` | 6-8 | 追求最快速度，可能触发限制 |

### 注意事项

1. 单线程模式最稳定，但速度最慢
2. 多线程模式在避免 baostock 限制的同时提供一定并发性
3. 多进程模式速度最快，但可能触发 baostock 限制
4. 建议先使用测试模式验证当前网络环境下的最佳配置

## 总结

本 notebook 提供了改进版的每日交易系统运行环境，支持三种并发模式：

1. **单线程模式**：最稳定，适合 baostock 访问限制严重时使用
2. **多线程模式**：在避免限制的同时提供一定并发性
3. **多进程模式**：速度最快，适合网络环境良好时使用

可以根据实际情况选择合适的并发模式，平衡稳定性和性能。